# Layered Yelp Review Prediction — Pretrained Embeddings + Soft Routing

This script implements a **two-layer modeling approach** for predicting Yelp star ratings:
1. **Sentiment Model:** Predicts coarse-grained sentiment (negative / neutral / positive)
2. **Star Rating Model:** Uses both pretrained embeddings and *soft sentiment probabilities* to predict the detailed 1–5 star rating. By using **pretrained sentence embeddings** (`all-MiniLM-L6-v2`) from `sentence-transformers` and **soft routing** (passing sentiment probabilities rather than
hard labels), this model captures nuanced sentiment signals while avoiding error propagation.

### Pipeline Overview
- Load cleaned Yelp data
- Encode text using pretrained sentence embeddings
- Train a logistic regression sentiment classifier
- Append sentiment probabilities to embeddings
- Train a second logistic regression for 5-star rating prediction

### Output
- Sentiment classifier performance
- Star rating classifier performance

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [2]:
# import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer

In [4]:
! pip install sentence_transformers

  Obtaining dependency information for tf-keras from https://files.pythonhosted.org/packages/85/6b/d9a8202bfe5c9e3b078cf550bafab962aa9d6b1a1f1180f0065399d4c9b2/tf_keras-2.20.1-py3-none-any.whl.metadata
  Obtaining dependency information for tensorflow<2.21,>=2.20 from https://files.pythonhosted.org/packages/ef/69/de33bd90dbddc8eede8f99ddeccfb374f7e18f84beb404bfe2cbbdf8df90/tensorflow-2.20.0-cp311-cp311-macosx_12_0_arm64.whl.metadata
  Obtaining dependency information for tensorboard~=2.20.0 from https://files.pythonhosted.org/packages/9c/d9/a5db55f88f258ac669a92858b70a714bbbd5acd993820b41ec4a96a4d77f/tensorboard-2.20.0-py3-none-any.whl.metadata
  Obtaining dependency information for keras>=3.10.0 from https://files.pythonhosted.org/packages/ba/61/cc8be27bd65082440754be443b17b6f7c185dec5e00dfdaeab4f8662e4a8/keras-3.12.0-py3-none-any.whl.metadata
  Obtaining dependency information for numpy>=1.26.0 from https://files.pythonhosted.org/packages/e4/04/ff11611200acd602a1e5129e36cfd25bf01ad8e

In [3]:
def load_data():
    folder = "../Preprocessing-FeatureExtraction/cleaned-data/"
    csv_files = glob.glob(os.path.join(folder, "*.csv"))

    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df["state"] = os.path.splitext(os.path.basename(file))[0]
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    df = data.dropna()

    # Sentiment: 0=negative, 1=neutral, 2=positive
    df["sentiment"] = df["stars"].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

    # Star label (0–4 instead of 1–5)
    df["label"] = df["stars"] - 1

    print(f"Loaded {len(df)} reviews from {len(dfs)} states.")
    return df

In [4]:
# Split data
df = load_data()
sample = df.sample(500000, random_state=42)

X = sample["clean_text"].tolist()
y_sentiment = sample["sentiment"].tolist()
y_stars = sample["label"].tolist()

X_train, X_test, y_train_sent, y_test_sent, y_train_star, y_test_star = train_test_split(
    X, y_sentiment, y_stars, test_size=0.1, random_state=42, stratify=y_sentiment
)

/var/folders/cl/5mfvhcls2nv2h9gy15m04b640000gn/T/ipykernel_7098/1017195081.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["sentiment"] = df["stars"].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))
/var/folders/cl/5mfvhcls2nv2h9gy15m04b640000gn/T/ipykernel_7098/1017195081.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = df["stars"] - 1


Loaded 5222860 reviews from 20 states.


In [5]:
# Sentence embeddings
print("Encoding text with SentenceTransformer embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = embedder.encode(X_train, show_progress_bar=True, batch_size=128)
X_test_emb = embedder.encode(X_test, show_progress_bar=True, batch_size=128)

print(f"Embedding shape: {X_train_emb.shape}")

Encoding text with SentenceTransformer embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3516 [00:00<?, ?it/s]

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding shape: (450000, 384)


In [6]:
# Layer 1 — Sentiment Classifier
print("\nTraining sentiment classifier...")
sentiment_clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
sentiment_clf.fit(X_train_emb, y_train_sent)

y_pred_sent = sentiment_clf.predict(X_test_emb)
print("\n=== Sentiment Classification Report ===")
print(classification_report(y_test_sent, y_pred_sent, digits=3))
print(f"Sentiment Accuracy: {accuracy_score(y_test_sent, y_pred_sent):.4f}")

# Get soft sentiment probabilities
train_sent_probs = sentiment_clf.predict_proba(X_train_emb)  # (n_train, 3)
test_sent_probs = sentiment_clf.predict_proba(X_test_emb)    # (n_test, 3)


Training sentiment classifier...

=== Sentiment Classification Report ===
              precision    recall  f1-score   support

           0      0.770     0.768     0.769     10241
           1      0.294     0.602     0.395      5605
           2      0.950     0.787     0.861     34154

    accuracy                          0.762     50000
   macro avg      0.671     0.719     0.675     50000
weighted avg      0.839     0.762     0.790     50000

Sentiment Accuracy: 0.7624


In [7]:
# Layer 2 — Star Rating Classifier
print("\nTraining star rating classifier with soft sentiment routing...")
alpha = 1.0  # weight for sentiment features — can tune

X_train_star = np.hstack([X_train_emb, alpha * train_sent_probs])
X_test_star = np.hstack([X_test_emb, alpha * test_sent_probs])

star_clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
star_clf.fit(X_train_star, y_train_star)

y_pred_star = star_clf.predict(X_test_star)
print("\n=== Star Rating Classification Report ===")
print(classification_report(y_test_star, y_pred_star, digits=3))
print(f"Star Rating Accuracy: {accuracy_score(y_test_star, y_pred_star):.4f}")


Training star rating classifier with soft sentiment routing...

=== Star Rating Classification Report ===
              precision    recall  f1-score   support

         0.0      0.689     0.733     0.711      6070
         1.0      0.334     0.444     0.381      4171
         2.0      0.322     0.428     0.368      5605
         3.0      0.450     0.421     0.435     11974
         4.0      0.787     0.686     0.733     22180

    accuracy                          0.579     50000
   macro avg      0.517     0.543     0.526     50000
weighted avg      0.604     0.579     0.589     50000

Star Rating Accuracy: 0.5794
